# CausaSent ABSA — Kaggle training notebook (v2: ATE + ABSA + contrastive)

Same shape as `kaggle_train.ipynb`: clone repo → install deps → HF auth → fetch dataset from HF Hub → train via `python -m src.train.train_phobert` → eval → push checkpoint back to HF.

All training code lives in `src/` and `configs/phobert.yaml`; this notebook is a thin pipeline that runs them on Kaggle.

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 16 GB or T4 ×2 30 GB).
2. **Settings → Internet → On** (needed for `pip install`, HF Hub, VnCoreNLP download).
3. **Add-ons → Secrets → add `HF_TOKEN`** with *write* access. Get at https://huggingface.co/settings/tokens.
4. The dataset must already be pushed to HF Hub. From your local repo run once:
   ```bash
   python scripts/push_dataset.py --source data/processed/ate --repo-id Tamir39/causasent-ate-v2
   ```

**Pipeline:**
1. Clone the `feat/skeleton` branch into `/kaggle/working/CausaSent`.
2. `pip install -r requirements.txt`.
3. Authenticate HF via the Kaggle secret.
4. Fetch the ABSA dataset from `Tamir39/causasent-ate-v2` → `data/processed/ate/{train,val,test}.json`.
5. Sanity-check env.
6. Download VnCoreNLP word-segmentation models (idempotent).
7. Train PhoBERT-large two-head tagger via `python -m src.train.train_phobert --config configs/phobert.yaml`.
8. Eval on the test split.
9. Push the best checkpoint to `Tamir39/causasent-phobert-ate`.

In [ ]:
# Cell 1 — Clone the repo into /kaggle/working.
import os, subprocess

REPO_URL = 'https://github.com/tamir39/causa-sent.git'
REPO_DIR = '/kaggle/working/CausaSent'
BRANCH = 'feat/skeleton'  # switch to 'develop' once merged

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])

os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# Cell 2 — Install dependencies.
%pip install -q -r requirements.txt 2>&1 | tail -10

In [ ]:
# Cell 3 — Authenticate with HF Hub via the Kaggle secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4 — Path + env setup.
# Keep dataset *inside* the repo dir so configs/*.yaml relative paths just work.
import sys
ROOT = '/kaggle/working/CausaSent'
os.environ['CAUSASENT_DATA_DIR'] = f'{ROOT}/data'
os.environ['MPLBACKEND'] = 'Agg'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5 — Pull the ABSA dataset from HF Hub into data/processed/ate/.
# Files land at <dest>/{train,val,test}.json (matches configs/phobert.yaml paths).
ATE_REPO = 'Tamir39/causasent-ate-v2'
ATE_DEST = f'{ROOT}/data/processed/ate'
!python scripts/fetch_dataset.py --repo-id $ATE_REPO --dest $ATE_DEST --force

import json
for split in ('train', 'val', 'test'):
    p = f'{ATE_DEST}/{split}.json'
    if not os.path.exists(p):
        print(f'  MISSING: {p}')
        continue
    data = json.load(open(p, encoding='utf-8'))
    anns = sum(len(r.get('annotations', [])) for r in data)
    print(f'  {split}: {len(data)} reviews / {anns} annotations')

In [ ]:
# Cell 6 — GPU + library sanity check.
!python scripts/check_env.py

In [ ]:
# Cell 7 — Pre-download VnCoreNLP word-segmentation models (idempotent).
# src/data/segmenter.py lazy-loads this; pre-downloading avoids per-process churn.
import py_vncorenlp
from pathlib import Path
VNCORENLP_DIR = Path(ROOT) / 'vncorenlp'
VNCORENLP_DIR.mkdir(exist_ok=True)
if not any(VNCORENLP_DIR.glob('VnCoreNLP-*.jar')):
    print('Downloading VnCoreNLP...')
    py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
print('VnCoreNLP ready at', VNCORENLP_DIR)

In [ ]:
# Cell 8 — Train PhoBERT-large two-head tagger (ATE + binary sentiment) + contrastive loss.
# Config: configs/phobert.yaml — paths trỏ data/processed/ate/*, contrastive_weight=0.1.
# Best checkpoint by mean(ate_f1, sent_acc) on val → checkpoints/phobert/best.pt.
# Run metadata → runs/<ts>-phobert/{config.yaml,git_sha.txt,env.txt,metrics.jsonl}.
!python -m src.train.train_phobert --config configs/phobert.yaml

In [ ]:
# Cell 9 — Eval on the test split (entity-F1 + sentiment accuracy/F1).
!python -m src.eval.eval_phobert --config configs/phobert.yaml --ckpt checkpoints/phobert/best.pt --split test

In [ ]:
# Cell 10 — Push best checkpoint + run metadata to HF Hub.
from pathlib import Path
from huggingface_hub import HfApi, create_repo

REPO_ID = 'Tamir39/causasent-phobert-ate'
LOCAL = 'checkpoints/phobert'

if Path(LOCAL).is_dir():
    create_repo(REPO_ID, repo_type='model', exist_ok=True, private=False)
    HfApi().upload_folder(
        folder_path=LOCAL,
        repo_id=REPO_ID,
        repo_type='model',
        commit_message='ATE + ABSA + contrastive training run from Kaggle',
        ignore_patterns=['**/__pycache__/**', '*.tmp'],
    )
    print(f'pushed -> https://huggingface.co/{REPO_ID}')
else:
    print(f'skip: no checkpoints at {LOCAL}')